In [1]:
!pip install --user matplotlib


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Kusoma faili letu la CSV tuliloliweka kwenye folda ya kazi
df = pd.read_csv('california_housing.csv')

# Kuangalia mistari 5 ya kwanza ya dataset
print("Mistari 5 ya kwanza:")
display(df.head())

# Kuangalia ukubwa wa data (Rows na Columns)
print("\nUkubwa wa Dataset (Rows, Columns):", df.shape)

# Kuangalia kama kuna data zilizopotea (Missing Values)
print("\nIdadi ya Missing Values kila column:")
print(df.isnull().sum())

Mistari 5 ya kwanza:


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25



Ukubwa wa Dataset (Rows, Columns): (20640, 8)

Idadi ya Missing Values kila column:
MedInc        0
HouseAge      0
AveRooms      0
AveBedrms     0
Population    0
AveOccup      0
Latitude      0
Longitude     0
dtype: int64


In [3]:
from sklearn.model_selection import train_test_split

# 1. Kugawanya mara ya kwanza: Train+Val (80%) na Test (20%)
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

# 2. Kugawanya Train+Val kuwa Train (60%) na Validation (20%)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42) # 0.25 ya 80% inatoa 20% ya zote

# 3. Kuweka Index vizuri
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# 4. Kutenganisha Target Variable yetu (Ambayo hapa ni MedInc - wastani wa kipato/bei ya eneo husika)
y_train = df_train['MedInc'].values
y_val = df_val['MedInc'].values
y_test = df_test['MedInc'].values

# Kuondoa MedInc kwenye dataframe za mafunzo ili modeli isiidanganye
del df_train['MedInc']
del df_val['MedInc']
del df_test['MedInc']

print("Ukubwa wa Train set:", df_train.shape)
print("Ukubwa wa Validation set:", df_val.shape)
print("Ukubwa wa Test set:", df_test.shape)

Ukubwa wa Train set: (12384, 7)
Ukubwa wa Validation set: (4128, 7)
Ukubwa wa Test set: (4128, 7)


In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# 1. Kuanzisha modeli ya Linear Regression
model = LinearRegression()

# 2. Kufunza modeli kwa kutumia Train set
model.fit(df_train, y_train)

# 3. Kufanya utabiri kwenye Validation set
y_pred = model.predict(df_val)

# 4. Kupima makosa (RMSE - Root Mean Squared Error)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print("RMSE ya Linear Regression kwenye Validation set ni:", rmse)

RMSE ya Linear Regression kwenye Validation set ni: 1.3101677351498193


In [5]:
from sklearn.ensemble import RandomForestRegressor

# 1. Kuanzisha modeli ya Random Forest
rf = RandomForestRegressor(n_estimators=10, random_state=42)

# 2. Kufunza modeli
rf.fit(df_train, y_train)

# 3. Kufanya utabiri kwenye Validation set
y_pred_rf = rf.predict(df_val)

# 4. Kupima RMSE ya Random Forest
rmse_rf = np.sqrt(mean_squared_error(y_val, y_pred_rf))

print("RMSE ya Random Forest kwenye Validation set ni:", rmse_rf)

RMSE ya Random Forest kwenye Validation set ni: 0.9254681470194431


In [6]:
# Jaribu kuongeza idadi ya miti kuwa 50 au 100 badala ya 10
rf_tuned = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42)

rf_tuned.fit(df_train, y_train)
y_pred_tuned = rf_tuned.predict(df_val)
rmse_tuned = np.sqrt(mean_squared_error(y_val, y_pred_tuned))

print("RMSE ya Random Forest baada ya Tuning ni:", rmse_tuned)

RMSE ya Random Forest baada ya Tuning ni: 0.931262917173746


In [7]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

print("1. Inasoma data...")
df = pd.read_csv('california_housing.csv')

# Kugawanya data
print("2. Inagawanya data...")
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

y_train = df_train['MedInc'].values
y_val = df_val['MedInc'].values

del df_train['MedInc']
del df_val['MedInc']

print("3. Inafunza modeli ya Random Forest...")
model = RandomForestRegressor(n_estimators=10, random_state=42)
model.fit(df_train, y_train)

# Kupima ubora kwenye Validation set
y_pred = model.predict(df_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(f"Modeli imekamilika! RMSE kwenye Validation ni: {rmse:.4f}")

# Kuhifadhi modeli kwa kutumia Pickle
output_file = 'model.bin'
with open(output_file, 'wb') as f_out:
    pickle.dump(model, f_out)

print(f"Modeli imehifadhiwa kwa mafanikio kwenye faili la: {output_file}")

1. Inasoma data...
2. Inagawanya data...
3. Inafunza modeli ya Random Forest...
Modeli imekamilika! RMSE kwenye Validation ni: 0.9255
Modeli imehifadhiwa kwa mafanikio kwenye faili la: model.bin
